# Capstone Project 3: Predictive Maintenance

This notebook includes the interim analysis and the final deployment work.
We first explore the data and compare three models. We then create the scripts,
Streamlit app and GitHub Actions workflow using the same approach as the course examples.

## Business Context

Unexpected engine breakdowns can lead to expensive repairs, downtime, and safety risks.
Sensor readings can help identify engines that may need further inspection. The supplied
project describes applications in vehicles, generators, and other engine-powered equipment.

## Objective

Build and compare classification models that predict the recorded engine condition from
engine speed, pressure, and temperature readings. Use the results to suggest the next steps
for a maintenance-screening prototype.

## Data Description

| Column | Description | Unit / meaning |
|---|---|---|
| Engine rpm | Engine speed | RPM |
| Lub oil pressure | Lubricating oil pressure | Bar or kPa; exact unit not confirmed |
| Fuel pressure | Fuel supply pressure | Bar or kPa; exact unit not confirmed |
| Coolant pressure | Cooling-system pressure | Bar or kPa; exact unit not confirmed |
| lub oil temp | Lubricating oil temperature | Celsius, according to the supplied description |
| Coolant temp | Coolant temperature | Celsius, according to the supplied description |
| Engine Condition | Target variable | Classes 0 and 1; the supplied mapping is ambiguous |

We use **class 0** and **class 1** rather than assigning normal/faulty labels without confirmation.
Class 1 is used as the positive label when calculating precision, recall, and F1-score.
The original external data source, engine IDs, and timestamps were not supplied. This notebook
classifies recorded conditions; it does not establish how far in advance a failure can be predicted.

## Installing the Required Libraries

Use Python 3.11. Install the packages from `requirements-training.txt` before running
this notebook. After installation, restart the kernel and run the cells in order.

```python
%pip install -r "requirements-training.txt"
```

In [ ]:
import sys
print("Python version:", sys.version)
print("Run the notebook with the python-starter Python 3.11 environment.")

## Importing the Necessary Libraries

Select a Python environment with pandas, numpy, matplotlib, seaborn, scikit-learn, joblib,
mlflow-skinny, sqlalchemy, and alembic installed. All analysis code is in this notebook.

In [ ]:
# For reading and exploring the data
import pandas as pd
import numpy as np
from pathlib import Path
import os
import json
import hashlib
import zipfile

In [ ]:
# For visualizing the data
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.float_format", lambda x: f"{x:.2f}")
pd.set_option("display.max_colwidth", None)
plt.rcParams.update({"axes.labelsize": 12, "axes.titlesize": 12,
                     "xtick.labelsize": 11, "ytick.labelsize": 11})

In [ ]:
# For splitting, preprocessing, and tuning the models
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone

In [ ]:
# For model evaluation, saving the model, and tracking experiments
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)
import joblib
import mlflow

## 1. Data Registration

The master folder is `Capstone Project`, and the CSV is in its `data` subfolder.
The following cells load the file and check its expected columns and values.

In [ ]:
# Locate the project folder when running from the project or repository directory
project_folder = Path.cwd()
if not (project_folder / "data" / "engine_data.csv").exists():
    project_folder = project_folder / "Capstone Project"

data_path = project_folder / "data" / "engine_data.csv"
engine_data = pd.read_csv(data_path)
data = engine_data.copy()

In [ ]:
expected_columns = ["Engine rpm", "Lub oil pressure", "Fuel pressure", "Coolant pressure",
                    "lub oil temp", "Coolant temp", "Engine Condition"]
target = "Engine Condition"
numeric_columns = expected_columns[:-1]

In [ ]:
# A small validation function acts as the registration check inside this notebook
def validate_data(df):
    if set(df.columns) != set(expected_columns):
        raise ValueError("The dataset columns do not match the expected columns.")
    if df.empty:
        raise ValueError("The dataset is empty.")
    for column in expected_columns:
        pd.to_numeric(df[column], errors="raise")
    if not df[target].isin([0, 1]).all() or df[target].nunique() != 2:
        raise ValueError("Engine Condition must contain both 0 and 1, with no missing labels.")
    if np.isinf(df[expected_columns].to_numpy(dtype=float)).any():
        raise ValueError("The dataset contains infinite values.")
    print("Dataset validation passed.")
    print("Rows:", df.shape[0], "| Columns:", df.shape[1])
    print("Missing cells:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print("Target counts:\n", df[target].value_counts().sort_index())

validate_data(data)

In [ ]:
# Record which version of the input file is being used
data_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
print("Dataset:", data_path.name)
print("SHA-256:", data_hash)

## 2. Exploratory Data Analysis
### Data Collection and Background

The file was supplied with the project. It contains engine speed, pressure, temperature,
and condition readings. The supplied brief explains the intended maintenance application,
but does not identify the actual collection method or sampling period.

### Observation

- The readings cover engine speed, lubrication, fuel delivery, and cooling, giving several views of engine operation.
- Without engine IDs or timestamps, we cannot establish that the findings apply to different engines or future operating periods.
- Repair costs and downtime outcomes are not provided, so the study cannot quantify maintenance savings.


### Data Overview
#### Viewing the First and Last Five Rows

In [ ]:
data.head()

In [ ]:
data.tail()

#### Checking the Shape and Data Types

In [ ]:
data.shape

In [ ]:
data.info()

#### Checking the Statistical Summary

In [ ]:
data.describe().T

### Observation

- Engine speed ranges from 61 to 2,239 RPM, so the observations cover a broad range of operating speeds.
- Average lubricating oil and coolant temperatures are 77.64 and 78.43 Celsius, respectively; these sample averages are not engineering operating limits.
- The maximum coolant reading of 195.53 Celsius warrants checking the sensor record and engine specifications before drawing a maintenance conclusion.


#### Checking Missing and Duplicate Values

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

#### Checking the Target Distribution

In [ ]:
class_summary = data[target].value_counts().sort_index().to_frame("Count")
class_summary["Percentage"] = class_summary["Count"] / len(data) * 100
class_summary

### Observation

- Class 1 represents 63.05% of the records and class 0 represents 36.95%; the sample contains more examples of one condition than the other.
- A model can appear accurate while recognizing the less common condition poorly, so maintenance decisions need a review of errors in both classes.
- These proportions cannot be interpreted as failure rates until the meanings of class 0 and class 1 are confirmed.


### Keeping a Test Set Aside

We reserve 20% of the cleaned rows before exploring sensor relationships. The detailed plots
below use only the training rows. Section 3 saves these same partitions for model building.
Exact duplicates are removed before splitting so identical rows cannot occur in both sets.

In [ ]:
clean_data = data.drop_duplicates().copy()
train_data, test_data = train_test_split(
    clean_data, test_size=0.20, random_state=42, stratify=clean_data[target]
)
eda_data = train_data.copy()
print("Training rows:", len(train_data))
print("Test rows:", len(test_data))

### Univariate Analysis

We first examine each sensor separately using histograms and boxplots.

In [ ]:
plt.figure(figsize=(9, 9))
for i, column in enumerate(numeric_columns):
    plt.subplot(3, 2, i + 1)
    sns.histplot(data=eda_data, x=column, bins=30)
    plt.title(column)
    plt.ylabel("Training observations")
plt.suptitle("Figure 1. Distribution of the Six Sensors", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 8))
for i, column in enumerate(numeric_columns):
    plt.subplot(3, 2, i + 1)
    sns.boxplot(data=eda_data, x=column)
    plt.title(column)
plt.suptitle("Figure 2. Sensor Boxplots", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Count unusual readings using the 1.5 x IQR rule
outlier_summary = []
for column in numeric_columns:
    q1 = eda_data[column].quantile(0.25)
    q3 = eda_data[column].quantile(0.75)
    iqr = q3 - q1
    count = ((eda_data[column] < q1 - 1.5 * iqr) | (eda_data[column] > q3 + 1.5 * iqr)).sum()
    outlier_summary.append({"Sensor": column, "IQR-flagged rows": count,
                            "Flagged percentage": count / len(eda_data) * 100})
pd.DataFrame(outlier_summary)

### Observation

- RPM, fuel pressure, and lubricating oil temperature include relatively high readings compared with most observations.
- Coolant temperature reaches 195.53 Celsius in the training sample; an engine specialist should assess whether this reflects an operating event or a sensor issue.
- Unusual readings should not be discarded automatically because they may contain information relevant to engine condition.


### Bivariate Analysis

We compare each sensor with Engine Condition to look for differences between the two classes.

In [ ]:
plt.figure(figsize=(9, 9))
for i, column in enumerate(numeric_columns):
    plt.subplot(3, 2, i + 1)
    sns.boxplot(data=eda_data, x=target, y=column)
    plt.title(column + " by Condition")
plt.suptitle("Figure 3. Sensor Values by Engine Condition", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
eda_data.groupby(target)[numeric_columns].mean().T

In [ ]:
eda_data.corr()[target].drop(target).sort_values()

### Observation

- Average training RPM is 885.74 for class 0 and 737.49 for class 1, showing that the recorded conditions occur under different average operating speeds.
- The sensor distributions overlap between classes, so one RPM or pressure cutoff is unlikely to identify every condition correctly.
- The differences support examining several readings together, but they do not establish that changing a reading would prevent a breakdown.


### Multivariate Analysis

The heatmap and scatterplots help us look at relationships involving several variables.

In [ ]:
plt.figure(figsize=(11, 7))
sns.heatmap(eda_data.corr(), annot=True, fmt=".2f", cmap="Spectral", vmin=-1, vmax=1)
plt.title("Figure 4. Correlation Between Sensors and Engine Condition")
plt.tight_layout()
plt.show()

In [ ]:
# A smaller random sample keeps the scatterplot easy to read
plot_data = eda_data.sample(n=2000, random_state=42)
fig, axes = plt.subplots(2, 1, figsize=(8, 7.5))
sns.scatterplot(data=plot_data, x="Engine rpm", y="Fuel pressure", hue=target, alpha=0.4, ax=axes[0])
sns.scatterplot(data=plot_data, x="Engine rpm", y="lub oil temp", hue=target, alpha=0.4, ax=axes[1])
fig.suptitle("Figure 5. Relationships Between Sensors and Condition Class")
plt.tight_layout()
plt.show()

### Observation

- The scatterplots show overlapping condition classes even when two sensors are considered together, indicating that a simple visual rule will miss some cases.
- Sensor readings describe different aspects of operation; the analysis supports considering speed, pressure, and temperature together during further investigation.
- These observed relationships do not define safe operating ranges or maintenance thresholds; engine-specific limits are still required.


### EDA Insights

### Observation

- The sensor patterns provide a basis for testing a maintenance-screening aid, but they do not cleanly separate every recorded condition.
- Review unusually high readings with an engine specialist before treating them as faults or removing them as errors.
- Confirm class definitions and collect engine IDs, timestamps, and actual maintenance outcomes to make the findings more useful for maintenance planning.


## 3. Data Preparation
### Data Cleaning and Feature Selection

There are no missing values or exact duplicate rows in this file. We still apply the duplicate
removal step so the cleaning decision is explicit. There are no ID or unnamed index columns to drop.
Extreme readings are kept because we do not have confirmed engineering limits.

In [ ]:
print("Original shape:", data.shape)
print("Cleaned shape:", clean_data.shape)
print("Missing values after cleaning:", clean_data.isnull().sum().sum())
print("Columns retained:", clean_data.columns.tolist())
assert set(train_data.index).isdisjoint(test_data.index)

### Saving the Training and Testing Sets

The saved CSV files allow the next stage to load exactly the same partitions. On a normal rerun,
outputs are saved under `Capstone Project/artifacts`; an optional environment setting can redirect
runtime outputs without changing the input CSV.

In [ ]:
# This setting only changes where generated split, log, and model files are stored
output_folder = Path(os.environ.get("CAPSTONE_OUTPUT_DIR", str(project_folder / "artifacts")))
split_folder = output_folder / "splits"
split_folder.mkdir(parents=True, exist_ok=True)
train_data.to_csv(split_folder / "train.csv", index=False)
test_data.to_csv(split_folder / "test.csv", index=False)
print(f"Saved train.csv ({len(train_data):,} rows) and test.csv ({len(test_data):,} rows).")

In [ ]:
split_summary = pd.DataFrame({
    "Rows": [len(train_data), len(test_data)],
    "Class 1 percentage": [train_data[target].mean() * 100, test_data[target].mean() * 100]
}, index=["Training", "Testing"])
split_summary

### Preparing and Loading a Data Artifact

The next cells save the train and test CSV files in a ZIP and read them into a
consumer folder. This checks the local handoff. Later, GitHub Actions uploads and
downloads these same split files between the preparation and training jobs.

In [ ]:
artifact_path = output_folder / "train_test_splits.zip"
with zipfile.ZipFile(artifact_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(split_folder / "train.csv", arcname="train.csv")
    archive.write(split_folder / "test.csv", arcname="test.csv")

consumer_folder = output_folder / "downloaded_splits"
consumer_folder.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(artifact_path) as archive:
    for name in ["train.csv", "test.csv"]:
        (consumer_folder / name).write_bytes(archive.read(name))
print("Split archive created and both CSV files loaded into the consumer folder.")

In [ ]:
# Confirm that the consumer files match the producer files
for name in ["train.csv", "test.csv"]:
    assert (split_folder / name).read_bytes() == (consumer_folder / name).read_bytes()
print("Local artifact handoff verified.")

## 4. Model Building with Experimentation Tracking
### Loading the Data for Model Building

In [ ]:
train = pd.read_csv(consumer_folder / "train.csv")
test = pd.read_csv(consumer_folder / "test.csv")

X_train = train.drop(columns=target)
y_train = train[target]
X_test = test.drop(columns=target)
y_test = test[target]
print(X_train.shape, X_test.shape)

### Model Evaluation Function

As in the other learner projects, a small function keeps the metric calculation consistent.
Precision, recall, and F1 below use class 1. Class 0 recall and balanced accuracy help show whether
the model performs differently for the two classes.

In [ ]:
def model_performance(model, predictors, actual):
    predicted = model.predict(predictors)
    probability = model.predict_proba(predictors)[:, 1]
    return pd.DataFrame({
        "Accuracy": [accuracy_score(actual, predicted)],
        "Precision (class 1)": [precision_score(actual, predicted, zero_division=0)],
        "Recall (class 1)": [recall_score(actual, predicted, zero_division=0)],
        "Recall (class 0)": [recall_score(actual, predicted, pos_label=0, zero_division=0)],
        "F1 (class 1)": [f1_score(actual, predicted, zero_division=0)],
        "Macro F1": [f1_score(actual, predicted, average="macro", zero_division=0)],
        "Balanced accuracy": [balanced_accuracy_score(actual, predicted)],
        "ROC-AUC": [roc_auc_score(actual, probability)]
    })

### Baseline Model

The baseline always predicts the most frequent training class. A useful model should do better
than this simple rule. We leave the baseline test evaluation until after selecting the tuned model.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)
model_performance(baseline, X_train, y_train)

### Observation

- Always choosing the more common class gives about 63% accuracy while failing to recognize every class-0 observation.
- A high headline accuracy alone would therefore provide weak evidence that a screening tool can recognize both operating conditions.
- The useful comparison is whether a model improves recognition of both classes, especially the condition that will be confirmed to require maintenance.


### Hyperparameter Tuning

We compare Decision Tree, Random Forest, and Gradient Boosting. Five-fold stratified
cross-validation uses only the training data. The best model is selected by average CV ROC-AUC;
the test set is not used to choose the model or its parameters. Threshold 0.50 is kept throughout.

Median imputation is inside each pipeline. No scaling is required for these tree-based models.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "f1_macro": "f1_macro", "balanced_accuracy": "balanced_accuracy"}

### Decision Tree

In [ ]:
dt_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("model", DecisionTreeClassifier(random_state=42))])
dt_parameters = {"model__max_depth": [3, 5, 8],
                 "model__min_samples_leaf": [10, 50],
                 "model__class_weight": [None, "balanced"]}
dt_search = GridSearchCV(dt_pipeline, dt_parameters, scoring=scoring, refit="roc_auc",
                         cv=cv, n_jobs=2, error_score="raise", verbose=1)
dt_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:", dt_search.best_params_)
print("Best CV ROC-AUC:", round(dt_search.best_score_, 2))

### Random Forest

In [ ]:
rf_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("model", RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=1))])
rf_parameters = {"model__max_depth": [6, 12],
                 "model__min_samples_leaf": [5, 20],
                 "model__class_weight": [None, "balanced"]}
rf_search = GridSearchCV(rf_pipeline, rf_parameters, scoring=scoring, refit="roc_auc",
                         cv=cv, n_jobs=2, error_score="raise", verbose=1)
rf_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:", rf_search.best_params_)
print("Best CV ROC-AUC:", round(rf_search.best_score_, 2))

### Gradient Boosting

In [ ]:
gb_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("model", GradientBoostingClassifier(random_state=42))])
gb_parameters = {"model__n_estimators": [80, 160],
                 "model__learning_rate": [0.05, 0.10],
                 "model__max_depth": [1, 2]}
gb_search = GridSearchCV(gb_pipeline, gb_parameters, scoring=scoring, refit="roc_auc",
                         cv=cv, n_jobs=2, error_score="raise", verbose=1)
gb_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:", gb_search.best_params_)
print("Best CV ROC-AUC:", round(gb_search.best_score_, 2))

### Model Comparison and Selection

In [ ]:
searches = {"Decision Tree": dt_search, "Random Forest": rf_search, "Gradient Boosting": gb_search}
comparison = pd.DataFrame({
    "Model": list(searches),
    "CV ROC-AUC": [search.best_score_ for search in searches.values()],
    "Best parameters": [search.best_params_ for search in searches.values()]
}).sort_values("CV ROC-AUC", ascending=False)
comparison

In [ ]:
best_model_name = comparison.iloc[0]["Model"]
best_model = searches[best_model_name].best_estimator_
print("Selected model:", best_model_name)

### Observation

- Gradient Boosting is the strongest offline candidate among the three model families tested, based on the training-data comparison.
- Its advantage in that comparison identifies a candidate for further assessment, rather than proving that it will reduce downtime or maintenance costs.
- Operational suitability depends on the held-out errors, confirmed class meanings, and the consequences of acting on an incorrect prediction.


### Experimentation Tracking

We use a local MLflow database, without a server or public account. Every tuned candidate gets
its own run. Parameters, CV metrics, and candidate results are also saved in a CSV for easy review.

In [ ]:
output_folder.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri("sqlite:///" + str((output_folder / "mlflow.db").resolve()))
client = mlflow.tracking.MlflowClient()
if client.get_experiment_by_name("capstone-predictive-maintenance") is None:
    client.create_experiment("capstone-predictive-maintenance", artifact_location=(output_folder / "mlflow_artifacts").resolve().as_uri())
mlflow.set_experiment("capstone-predictive-maintenance")

In [ ]:
# Log every tried combination, not just the winning parameters
experiment_results = []
with mlflow.start_run(run_name="Model comparison") as parent_run:
    mlflow.log_params({"random_state": 42, "cv_folds": 5, "source_sha256": data_hash})
    for model_name, search in searches.items():
        for i, parameters in enumerate(search.cv_results_["params"]):
            candidate_model = clone(search.estimator).set_params(**parameters)
            score = float(search.cv_results_["mean_test_roc_auc"][i])
            with mlflow.start_run(run_name=f"{model_name} candidate {i + 1}", nested=True):
                mlflow.log_param("model_name", model_name)
                mlflow.log_param("imputer_strategy", "median")
                mlflow.log_params(candidate_model.named_steps["model"].get_params())
                mlflow.log_metrics({"cv_roc_auc": score,
                                    "cv_auc_std": float(search.cv_results_["std_test_roc_auc"][i]),
                                    "cv_f1_macro": float(search.cv_results_["mean_test_f1_macro"][i]),
                                    "cv_balanced_accuracy": float(search.cv_results_["mean_test_balanced_accuracy"][i])})
            experiment_results.append({"Model": model_name, "Parameters": parameters,
                                       "CV ROC-AUC": score,
                                       "CV AUC SD": float(search.cv_results_["std_test_roc_auc"][i]),
                                       "CV macro F1": float(search.cv_results_["mean_test_f1_macro"][i]),
                                       "CV balanced accuracy": float(search.cv_results_["mean_test_balanced_accuracy"][i])})
print(f"Logged {len(experiment_results)} candidate runs with parameters and validation metrics.")

In [ ]:
experiment_results = pd.DataFrame(experiment_results)
experiment_results.to_csv(output_folder / "experiment_results.csv", index=False)
experiment_results

In [ ]:
experiment = client.get_experiment_by_name("capstone-predictive-maintenance")
runs = client.search_runs([experiment.experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'")
print("MLflow candidate runs:", len(runs))
assert len(runs) == len(experiment_results) == 28

### Evaluating the Selected Model

We now evaluate the selected model and baseline on the held-out test set.

In [ ]:
baseline_test = model_performance(baseline, X_test, y_test)
selected_train = model_performance(best_model, X_train, y_train)
selected_test = model_performance(best_model, X_test, y_test)
performance_comparison = pd.concat([baseline_test, selected_train, selected_test])
performance_comparison.index = ["Baseline - Test", "Selected Model - Train", "Selected Model - Test"]
performance_comparison
performance_comparison.to_csv(output_folder / "performance_comparison.csv")
performance_comparison

In [ ]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, digits=2))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Class 0", "Class 1"], cmap="Blues")
plt.title("Figure 6. Confusion Matrix - Selected Model")
plt.tight_layout()
plt.show()

In [ ]:
y_probability = best_model.predict_proba(X_test)[:, 1]
RocCurveDisplay.from_predictions(y_test, y_probability, name=best_model_name)
plt.plot([0, 1], [0, 1], "--", color="grey", label="Chance")
plt.title("Figure 7. ROC Curve - Selected Model")
plt.legend()
plt.tight_layout()
plt.show()

### Observation

- Test accuracy improves from 63.04% for the simple baseline to 66.78% for the selected model, a gain of 3.74 percentage points; this is not evidence of equivalent cost savings.
- The model recognizes 86.20% of class 1 but only 33.66% of class 0. If class 0 is the maintenance-critical condition, this weakness would be especially important.
- There are 958 class-0-to-1 errors and 340 class-1-to-0 errors; confirm which condition needs maintenance and the cost of each error before using predictions to trigger work.


### Feature Importance

Feature importance shows how the selected tree-based model uses the sensor columns. It does not establish a cause of failure.

In [ ]:
importance = pd.DataFrame({"Feature": X_train.columns,
                           "Importance": best_model.named_steps["model"].feature_importances_})
importance = importance.sort_values("Importance", ascending=False)
importance

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=importance, x="Importance", y="Feature", color="steelblue")
plt.title("Figure 8. Feature Importance - Selected Model")
plt.tight_layout()
plt.show()

### Observation

- RPM contributes most to the selected model, making engine-speed patterns a useful starting point for investigating its predictions.
- Fuel pressure and lubricating oil temperature are the next most influential readings, supporting a review of several engine measurements together.
- Model importance does not show what causes failure and should not be used by itself to prescribe a repair or set an alarm limit.


### Saving the Best Model

Save the fitted pipeline so preprocessing and classification remain together.
The Streamlit app will load this file. GitHub Actions will also train and commit
the model when the workflow runs.

In [ ]:
model_folder = output_folder / "models"
model_folder.mkdir(parents=True, exist_ok=True)
model_path = model_folder / "best_model.joblib"
joblib.dump(best_model, model_path)
print("Saved:", model_path.name)

In [ ]:
# Check that loading the saved model does not change its predictions
loaded_model = joblib.load(model_path)
np.testing.assert_array_equal(y_pred, loaded_model.predict(X_test))
print("Saved model predictions verified.")

In [ ]:
model_details = {"selected_model": best_model_name,
                 "parameters": searches[best_model_name].best_params_,
                 "feature_columns": X_train.columns.tolist(),
                 "source_sha256": data_hash,
                 "class_meanings": "Not confirmed"}
(model_folder / "model_details.json").write_text(json.dumps(model_details, indent=2))
print("Model details saved.")

## Business Insights and Recommendations

### Observation

- The selected model improves on the baseline, but weak recognition of class 0 limits confidence in its use as a maintenance-screening tool.
- Confirm the condition labels and pressure units, then work with an engine specialist to set an alert threshold using the consequences of missed conditions and unnecessary inspections.
- Validate on new engines and future readings, and measure confirmed faults, inspection workload, downtime, and repair costs before claiming operational benefits.


### Saving the Code as Scripts

The course examples use `%%writefile` to create Python scripts from the notebook.
We use the same approach below. The training script uses the three models and
parameter grids already tested above.

In [ ]:
(project_folder / "scripts").mkdir(exist_ok=True)
(project_folder.parent / ".github/workflows").mkdir(parents=True, exist_ok=True)
print("Script and workflow folders created.")

In [ ]:
%%writefile "{project_folder}/scripts/data_register.py"
from pathlib import Path
import pandas as pd

project_folder = Path(__file__).resolve().parents[1]
data = pd.read_csv(project_folder / "data/engine_data.csv")
expected_columns = ["Engine rpm", "Lub oil pressure", "Fuel pressure", "Coolant pressure",
                    "lub oil temp", "Coolant temp", "Engine Condition"]
if set(data.columns) != set(expected_columns):
    raise ValueError("The dataset does not have the expected columns.")
if not data["Engine Condition"].isin([0, 1]).all():
    raise ValueError("Engine Condition must contain 0 or 1.")
print("Dataset registered successfully.")
print("Rows and columns:", data.shape)
print("Missing values:\n", data.isnull().sum())
print("Condition counts:\n", data["Engine Condition"].value_counts())


In [ ]:
%%writefile "{project_folder}/scripts/prep.py"
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

project_folder = Path(__file__).resolve().parents[1]
data = pd.read_csv(project_folder / "data/engine_data.csv")
data = data.drop_duplicates()
# All six sensor columns are useful, so no column is removed.
train, test = train_test_split(data, test_size=0.20, random_state=42,
                               stratify=data["Engine Condition"])
split_folder = project_folder / "artifacts/splits"
split_folder.mkdir(parents=True, exist_ok=True)
train.to_csv(split_folder / "train.csv", index=False)
test.to_csv(split_folder / "test.csv", index=False)
print("Training rows:", len(train))
print("Testing rows:", len(test))


In [ ]:
%%writefile "{project_folder}/scripts/train.py"
from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np
import joblib
import mlflow
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, roc_auc_score, classification_report)

project_folder = Path(__file__).resolve().parents[1]
output_folder = project_folder / "artifacts"
train = pd.read_csv(output_folder / "splits/train.csv")
test = pd.read_csv(output_folder / "splits/test.csv")
target = "Engine Condition"
X_train = train.drop(columns=target)
y_train = train[target]
X_test = test.drop(columns=target)
y_test = test[target]
data_hash = hashlib.sha256((project_folder / "data/engine_data.csv").read_bytes()).hexdigest()

def model_performance(model, predictors, actual):
    predicted = model.predict(predictors)
    probability = model.predict_proba(predictors)[:, 1]
    return pd.DataFrame({
        "Accuracy": [accuracy_score(actual, predicted)],
        "Precision (class 1)": [precision_score(actual, predicted, zero_division=0)],
        "Recall (class 1)": [recall_score(actual, predicted, zero_division=0)],
        "Recall (class 0)": [recall_score(actual, predicted, pos_label=0, zero_division=0)],
        "F1 (class 1)": [f1_score(actual, predicted, zero_division=0)],
        "Macro F1": [f1_score(actual, predicted, average="macro", zero_division=0)],
        "Balanced accuracy": [balanced_accuracy_score(actual, predicted)],
        "ROC-AUC": [roc_auc_score(actual, probability)]
    })

baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)
model_performance(baseline, X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "f1_macro": "f1_macro", "balanced_accuracy": "balanced_accuracy"}

dt_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("model", DecisionTreeClassifier(random_state=42))])
dt_parameters = {"model__max_depth": [3, 5, 8],
                 "model__min_samples_leaf": [10, 50],
                 "model__class_weight": [None, "balanced"]}
dt_search = GridSearchCV(dt_pipeline, dt_parameters, scoring=scoring, refit="roc_auc",
                         cv=cv, n_jobs=2, error_score="raise", verbose=1)
dt_search.fit(X_train, y_train)

print("Best parameters:", dt_search.best_params_)
print("Best CV ROC-AUC:", round(dt_search.best_score_, 2))

rf_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("model", RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=1))])
rf_parameters = {"model__max_depth": [6, 12],
                 "model__min_samples_leaf": [5, 20],
                 "model__class_weight": [None, "balanced"]}
rf_search = GridSearchCV(rf_pipeline, rf_parameters, scoring=scoring, refit="roc_auc",
                         cv=cv, n_jobs=2, error_score="raise", verbose=1)
rf_search.fit(X_train, y_train)

print("Best parameters:", rf_search.best_params_)
print("Best CV ROC-AUC:", round(rf_search.best_score_, 2))

gb_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("model", GradientBoostingClassifier(random_state=42))])
gb_parameters = {"model__n_estimators": [80, 160],
                 "model__learning_rate": [0.05, 0.10],
                 "model__max_depth": [1, 2]}
gb_search = GridSearchCV(gb_pipeline, gb_parameters, scoring=scoring, refit="roc_auc",
                         cv=cv, n_jobs=2, error_score="raise", verbose=1)
gb_search.fit(X_train, y_train)

print("Best parameters:", gb_search.best_params_)
print("Best CV ROC-AUC:", round(gb_search.best_score_, 2))

searches = {"Decision Tree": dt_search, "Random Forest": rf_search, "Gradient Boosting": gb_search}
comparison = pd.DataFrame({
    "Model": list(searches),
    "CV ROC-AUC": [search.best_score_ for search in searches.values()],
    "Best parameters": [search.best_params_ for search in searches.values()]
}).sort_values("CV ROC-AUC", ascending=False)
comparison

best_model_name = comparison.iloc[0]["Model"]
best_model = searches[best_model_name].best_estimator_
print("Selected model:", best_model_name)

output_folder.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri("sqlite:///" + str((output_folder / "mlflow.db").resolve()))
client = mlflow.tracking.MlflowClient()
if client.get_experiment_by_name("capstone-predictive-maintenance") is None:
    client.create_experiment("capstone-predictive-maintenance", artifact_location=(output_folder / "mlflow_artifacts").resolve().as_uri())
mlflow.set_experiment("capstone-predictive-maintenance")

# Log every tried combination, not just the winning parameters
experiment_results = []
with mlflow.start_run(run_name="Model comparison") as parent_run:
    mlflow.log_params({"random_state": 42, "cv_folds": 5, "source_sha256": data_hash})
    for model_name, search in searches.items():
        for i, parameters in enumerate(search.cv_results_["params"]):
            candidate_model = clone(search.estimator).set_params(**parameters)
            score = float(search.cv_results_["mean_test_roc_auc"][i])
            with mlflow.start_run(run_name=f"{model_name} candidate {i + 1}", nested=True):
                mlflow.log_param("model_name", model_name)
                mlflow.log_param("imputer_strategy", "median")
                mlflow.log_params(candidate_model.named_steps["model"].get_params())
                mlflow.log_metrics({"cv_roc_auc": score,
                                    "cv_auc_std": float(search.cv_results_["std_test_roc_auc"][i]),
                                    "cv_f1_macro": float(search.cv_results_["mean_test_f1_macro"][i]),
                                    "cv_balanced_accuracy": float(search.cv_results_["mean_test_balanced_accuracy"][i])})
            experiment_results.append({"Model": model_name, "Parameters": parameters,
                                       "CV ROC-AUC": score,
                                       "CV AUC SD": float(search.cv_results_["std_test_roc_auc"][i]),
                                       "CV macro F1": float(search.cv_results_["mean_test_f1_macro"][i]),
                                       "CV balanced accuracy": float(search.cv_results_["mean_test_balanced_accuracy"][i])})
print(f"Logged {len(experiment_results)} candidate runs with parameters and validation metrics.")

experiment_results = pd.DataFrame(experiment_results)
experiment_results.to_csv(output_folder / "experiment_results.csv", index=False)
experiment_results
print(experiment_results)

experiment = client.get_experiment_by_name("capstone-predictive-maintenance")
runs = client.search_runs([experiment.experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'")
print("MLflow candidate runs:", len(runs))
assert len(runs) == len(experiment_results) == 28

baseline_test = model_performance(baseline, X_test, y_test)
selected_train = model_performance(best_model, X_train, y_train)
selected_test = model_performance(best_model, X_test, y_test)
performance_comparison = pd.concat([baseline_test, selected_train, selected_test])
performance_comparison.index = ["Baseline - Test", "Selected Model - Train", "Selected Model - Test"]
performance_comparison
performance_comparison.to_csv(output_folder / "performance_comparison.csv")
print(performance_comparison)

y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, digits=2))
y_pred = best_model.predict(X_test)

model_folder = output_folder / "models"
model_folder.mkdir(parents=True, exist_ok=True)
model_path = model_folder / "best_model.joblib"
joblib.dump(best_model, model_path)
print("Saved:", model_path.name)

# Check that loading the saved model does not change its predictions
loaded_model = joblib.load(model_path)
np.testing.assert_array_equal(best_model.predict(X_test), loaded_model.predict(X_test))
print("Saved model predictions verified.")

model_details = {"selected_model": best_model_name,
                 "parameters": searches[best_model_name].best_params_,
                 "feature_columns": X_train.columns.tolist(),
                 "source_sha256": data_hash,
                 "class_meanings": "Not confirmed"}
(model_folder / "model_details.json").write_text(json.dumps(model_details, indent=2))
print("Model details saved.")


## 5. Model Deployment
### Streamlit App

The app loads the saved model, collects the six sensor readings in a dataframe,
and displays the predicted condition. It follows the Streamlit app in the sample
notebooks. The class meanings and pressure units are still unconfirmed.

In [ ]:
%%writefile "{project_folder}/app.py"
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

project_folder = Path(__file__).resolve().parent
model = joblib.load(project_folder / "artifacts/models/best_model.joblib")
st.title("Engine Condition Prediction")
st.write("Enter the six sensor readings to predict the recorded engine condition.")
st.info("The meaning of class 0/1 and the pressure units need to be confirmed with the data owner.")

engine_rpm = st.number_input("Engine rpm", value=746.0, min_value=0.0)
lub_oil_pressure = st.number_input("Lub oil pressure", value=3.16, min_value=0.0)
fuel_pressure = st.number_input("Fuel pressure", value=6.20, min_value=0.0)
coolant_pressure = st.number_input("Coolant pressure", value=2.17, min_value=0.0)
lub_oil_temp = st.number_input("Lub oil temp (Celsius)", value=76.82)
coolant_temp = st.number_input("Coolant temp (Celsius)", value=78.35)

input_data = pd.DataFrame([{
    "Engine rpm": engine_rpm,
    "Lub oil pressure": lub_oil_pressure,
    "Fuel pressure": fuel_pressure,
    "Coolant pressure": coolant_pressure,
    "lub oil temp": lub_oil_temp,
    "Coolant temp": coolant_temp
}])
if st.button("Predict"):
    st.write("Input readings:")
    st.dataframe(input_data)
    prediction = model.predict(input_data)[0]
    score = model.predict_proba(input_data)[0, 1]
    st.write("Predicted recorded condition: Class", int(prediction))
    st.write("Model score for class 1:", round(float(score), 2))


### App Dependencies

Streamlit Community Cloud installs the packages listed in `requirements.txt`,
which is in the same folder as `app.py`. The versions match the training environment.

In [ ]:
%%writefile "{project_folder}/requirements.txt"
# Runtime pins match the environment used to create the serialized model.
streamlit==1.66.1
numpy==2.4.6
pandas==3.0.6
scikit-learn==1.9.1
joblib==1.6.0


## 6. GitHub Actions Workflow

The workflow has three jobs: **Register Dataset -> Data Preparation -> Model Training**.
The preparation job uploads the train/test files. The training job downloads them,
tunes the models, logs the results to MLflow and commits the selected model.
The workflow file must be in `.github/workflows` at the repository root.

In [ ]:
%%writefile "{project_folder.parent}/.github/workflows/pipeline.yml"
name: Predictive Maintenance Pipeline

on:
  push:
    branches: [main]
    paths:
      - 'Capstone Project/data/**'
      - 'Capstone Project/scripts/data_register.py'
      - 'Capstone Project/scripts/prep.py'
      - 'Capstone Project/scripts/train.py'
      - 'Capstone Project/requirements-training.txt'
      - '.github/workflows/pipeline.yml'
  workflow_dispatch:

permissions:
  contents: read

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install -r 'Capstone Project/requirements-training.txt'
      - run: python 'Capstone Project/scripts/data_register.py'

  data-preparation:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install -r 'Capstone Project/requirements-training.txt'
      - run: python 'Capstone Project/scripts/prep.py'
      - uses: actions/upload-artifact@v4
        with:
          name: train-test-splits
          path: 'Capstone Project/artifacts/splits/'
          if-no-files-found: error

  model-training:
    needs: data-preparation
    runs-on: ubuntu-latest
    permissions:
      contents: write
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install -r 'Capstone Project/requirements-training.txt'
      - uses: actions/download-artifact@v4
        with:
          name: train-test-splits
          path: 'Capstone Project/artifacts/splits/'
      - run: python 'Capstone Project/scripts/train.py'
      - uses: actions/upload-artifact@v4
        with:
          name: training-results
          path: |
            Capstone Project/artifacts/experiment_results.csv
            Capstone Project/artifacts/performance_comparison.csv
            Capstone Project/artifacts/mlflow.db
            Capstone Project/artifacts/mlflow_artifacts/
      - name: Commit the trained model
        run: |
          git config user.name 'github-actions[bot]'
          git config user.email '41898282+github-actions[bot]@users.noreply.github.com'
          git add 'Capstone Project/artifacts/models/'
          if ! git diff --cached --quiet; then
            git commit -m 'Update trained model [skip ci]'
            git push origin HEAD:main
          fi


### Training Dependencies

These packages are used by the notebook and the GitHub Actions jobs.

In [ ]:
%%writefile "{project_folder}/requirements-training.txt"
-r requirements.txt
matplotlib==3.11.2
seaborn==0.13.2
mlflow-skinny==3.17.1
sqlalchemy
alembic
nbformat==5.11.1
nbclient==0.11.0
nbconvert==7.17.1
ipykernel
PyYAML==6.0.3
PyGithub==2.9.1


### GitHub Authentication and Upload

Save the notebook before running these cells. As in the course examples, PyGithub
uploads the project files to the confirmed repository. Enter a token in the masked
prompt. It needs **Contents: Read and write** and **Workflows: Read and write** for
`ggajayant/maintenance-prediction` (or classic token scopes `repo` and `workflow`).
Never paste the token into a code cell. The next cell shows each file being uploaded
and prints the GitHub error message if something fails.

In [167]:
from github import Github, Auth, GithubException
from getpass import getpass

REPO = "ggajayant/maintenance-prediction"
BRANCH = "main"
# Batch export skips the prompt. Interactive Run All asks for your token.
GH_TOKEN = None
if os.environ.get("CAPSTONE_NONINTERACTIVE") == "1":
    print("Automated local run only: GitHub upload has NOT been run.")
    print("Run this authentication cell and the upload cell interactively to publish.")
else:
    GH_TOKEN = getpass("Enter your GitHub token (masked): ").strip()
    if not GH_TOKEN:
        raise ValueError("No token was entered. Nothing has been uploaded.")
    print("Token entered. Run the next cell to upload the files.")

Token entered. Run the next cell to upload the files.


In [ ]:
files_to_upload = [
    "data/engine_data.csv", "scripts/data_register.py", "scripts/prep.py", "scripts/train.py",
    "app.py", "requirements.txt", "requirements-training.txt", "README.md",
    "artifacts/models/best_model.joblib", "artifacts/models/model_details.json",
    "Capstone Project.ipynb"
]
upload_ok = False
if not GH_TOKEN:
    print("NOT UPLOADED: run the authentication cell interactively first.")
else:
    gh = Github(auth=Auth.Token(GH_TOKEN), timeout=30)
    try:
        print("Connecting to", REPO, flush=True)
        repo = gh.get_repo(REPO)
        print("Connected to", repo.full_name, flush=True)
        # Upload the workflow last so the first run has all its required files.
        upload_paths = [(project_folder / name, "Capstone Project/" + name)
                        for name in files_to_upload]
        upload_paths.append((project_folder.parent / ".github/workflows/pipeline.yml",
                             ".github/workflows/pipeline.yml"))
        for local_path, repo_path in upload_paths:
            content = local_path.read_bytes()
            print("Uploading:", repo_path, flush=True)
            try:
                existing = repo.get_contents(repo_path, ref=BRANCH)
            except GithubException as error:
                if error.status != 404:
                    raise
                repo.create_file(repo_path, "Add " + repo_path, content, branch=BRANCH)
                print("  Added", flush=True)
            else:
                if existing.decoded_content == content:
                    print("  Already up to date", flush=True)
                else:
                    repo.update_file(repo_path, "Update " + repo_path, content,
                                     existing.sha, branch=BRANCH)
                    print("  Updated", flush=True)
        upload_ok = True
        print("Upload completed:", repo.html_url)
        print("Check the workflow at:", repo.html_url + "/actions")
    except GithubException as error:
        message = error.data.get("message", "GitHub request failed") if isinstance(error.data, dict) else "GitHub request failed"
        print("Upload stopped. HTTP status:", error.status)
        print("GitHub message:", message)
        print("Check that the token includes this repository and has contents/workflows write access.")
        raise RuntimeError("GitHub upload did not complete. See the message above.") from None
    finally:
        gh.close()
        GH_TOKEN = None


Connecting to ggajayant/maintenance-prediction
Connected to ggajayant/maintenance-prediction
Uploading: Capstone Project/data/engine_data.csv
  Added
Uploading: Capstone Project/scripts/data_register.py
  Added
Uploading: Capstone Project/scripts/prep.py
  Added
Uploading: Capstone Project/scripts/train.py
  Added
Uploading: Capstone Project/app.py
  Added
Uploading: Capstone Project/requirements.txt
  Added
Uploading: Capstone Project/requirements-training.txt
  Added
Uploading: Capstone Project/README.md


### Run the Workflow and Deploy the App

1. Open https://github.com/ggajayant/maintenance-prediction/actions and check the three jobs.
   The upload triggers the workflow; you can also select **Run workflow**.
2. In Streamlit Community Cloud, create an app using repository
   `ggajayant/maintenance-prediction`, branch `main`, and file `Capstone Project/app.py`.
3. Choose Python 3.11. After the app loads, enter readings and try a prediction.

A successful local notebook run does not mean these hosted steps have completed.

## 7. Output Evaluation

Repository: https://github.com/ggajayant/maintenance-prediction

Add the actual workflow run URL, repository/workflow screenshots, Streamlit app URL
and app screenshot after checking them. Do not replace these with local results.

In [ ]:
print("Local training complete:", best_model_name)
print("Model file:", model_path)
print("GitHub upload in this run:", "Completed" if upload_ok else "Not completed")
print("Hosted workflow and Streamlit evidence: still to be verified.")

## 8. Actionable Insights and Recommendations

- The model improves accuracy from 63.04% to 66.78%, but it still makes many errors.
- Class 0 recall is only 33.66%. Confirm which class needs maintenance before using the result.
- Check pressure units and unusual readings with the data owner.
- Collect engine IDs, timestamps and maintenance records to test on new engines and future readings.
- Try the app alongside existing inspections and compare the results before relying on it.
- Check the GitHub workflow and Streamlit deployment and add the actual links and screenshots.

## References

1. User-provided `engine_data.csv` and the business context/data description below. The original collection source was not supplied.
2. Great Learning, *Capstone Project 3 - Predictive Maintenance - Guidelines*, supplied local PDF, for submission requirements.
3. [scikit-learn: GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), for parameter tuning.
4. [scikit-learn: Classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html), for model evaluation.
5. [MLflow: Experiment tracking](https://mlflow.org/docs/latest/ml/tracking/), for parameter and metric logging.

Charts and numerical findings in this notebook are based on the supplied data. The documentation
references explain the methods; they are not presented as the dataset's original source.

## Appendix: Original Project Brief and Rubric

The original project brief is retained below. This notebook now includes both the
analysis and the final deployment sections.

Business Context:

Vehicle breakdowns and engine failures lead to significant financial losses for both individual owners and fleet operators. Unexpected engine failures can cause expensive repairs, operational downtime, and safety risks. Predictive maintenance in the automotive industry can help minimize these issues by leveraging sensor data to forecast potential failures before they occur.

Automobile manufacturers, fleet managers, and service providers aim to develop data-driven solutions to improve engine reliability and optimize maintenance schedules. By analyzing engine health parameters such as RPM, temperature, pressure, and other sensor readings, machine learning models can be trained to predict when an engine requires maintenance, allowing proactive intervention before a failure occurs.

The sensor values in the dataset are consistent with the operating parameters of larger and small engines commonly found in equipment like Vechiles, lawnmowers, portable generators, and compact machinery. Some engines operate at lower RPMs, pressures, and temperatures compared to larger automotive engines and vice versa. Therefore, the data is appropriate for developing predictive maintenance models tailored to large and small engine applications.

Objective:

As a Data Scientist, your goal is to build a predictive maintenance model that can analyze historical and real-time engine sensor data to identify potential failures. The model should accurately classify whether an engine requires maintenance or is operating normally.

This solution will help:

Reduce unplanned breakdowns and costly repairs.
Improve vehicle performance and engine lifespan.
Optimize maintenance schedules to minimize downtime
Provide data-driven insights to manufacturers and fleet operators for better decision-making.
Approach:

To achieve this objective, you will:

Explore and preprocess the dataset, handling missing values, outliers, and sensor anomalies.
Perform exploratory data analysis (EDA) to identify patterns in engine health parameters.
Develop machine learning models (e.g., Decision Trees, Random Forest, XGBoost, or Deep Learning models) to predict engine failures.
Evaluate model performance using appropriate classification metrics such as accuracy, precision, recall, and F1-score.
Deploy the model in a real-time monitoring system to alert users about potential failures.
By implementing a robust predictive maintenance system, this solution can lead to significant cost savings and improved efficiency in the automotive sector.

Data Description:

Engine_RPM: The number of revolutions per minute (RPM) of the engine, indicating engine speed. It is defined in Revolutions per Minute (RPM).
Lub_Oil_Pressure: The pressure of the lubricating oil in the engine, essential for reducing friction and wear. It is defined in bar or kilopascals (kPa)
Fuel_Pressure: The pressure at which fuel is supplied to the engine, critical for proper combustion. It is defined in bar or kilopascals (kPa)
Coolant_Pressure: The pressure of the engine coolant, affecting engine temperature regulation. It is defined in bar or kilopascals (kPa)
Lub_Oil_Temperature: The temperature of the lubricating oil, which impacts viscosity and engine performance. It is defined in degrees Celsius (°C)
Coolant_Temperature: The temperature of the engine coolant, crucial for preventing overheating. It is defined in degrees Celsius (°C)
Engine_Condition: A categorical or numerical label representing the health of the engine, potentially indicating normal operation or various levels of wear and failure risks. It is defined as a categorical variable (0/1) representing a state such as "0 = Off/False/Active" and "1 = On/True/Faulty"
Evaluation Rubrics for Interim Report

Section	Description	Points
Data Registration	- Create a master folder and a subfolder "data", and add the dataset CSV to it
- Register the dataset with a validation script that checks the expected columns and prints a summary

6
Exploratory Data Analysis	- Data collection and background
- Data overview

- Univariate analysis

- Bivariate analysis

- Multivariate analysis

- Insights/observations based on EDA

10
Data Preparation	- Load the dataset directly from the repository data folder.
- Perform data cleaning and remove any unnecessary columns.

- Split the cleaned dataset into training and testing sets, and save them locally.

- Make the train and test splits available to the next job as a workflow artifact.

10
Model Building with Experimentation Tracking	- Load the train and test data from the workflow artifact
- Define a model and parameters

- Tune the model with the defined parameters

- Log all the tuned parameters

- Evaluate the model performance

- Save the best model so the pipeline can commit it to the repository

* The ML models to be built can be any of the following algorithms, such as Decision Tree, Bagging, Random Forest, AdaBoost, Gradient Boosting, and XGBoost

8
Business Report Quality	- Adhere to the business report checklist	6
Total		40
Evaluation Rubrics for Final Report

Section	Description	Points
Data Registration	- Create a master folder and a subfolder "data", and add the dataset CSV to it
- Register the dataset with a validation script that checks the expected columns and prints a summary

2
Exploratory Data Analysis	- Data collection and background
- Data overview

- Univariate analysis

- Bivariate analysis

- Multivariate analysis

- Insights/observations based on EDA

3
Data preparation	- Load the dataset directly from the repository data folder.
- Perform data cleaning and remove any unnecessary columns.

- Split the cleaned dataset into training and testing sets, and save them locally.

- Make the train and test splits available to the next job as a workflow artifact.

4
Model Building with Experimentation Tracking	- Load the train and test data from the workflow artifact
- Define a model and parameters

- Tune the model with the defined parameters

- Log all the tuned parameters

- Evaluate the model performance

- Save the best model so the pipeline can commit it to the repository

* The ML models to be built can be any of the following algorithms, such as Decision Tree, Bagging, Random Forest, AdaBoost, Gradient Boosting, and XGBoost

6
Model Deployment	- Create a Streamlit app that loads the model committed to the repository
- Get the inputs and save them into a dataframe

- Define a dependencies file for the deployment

- Deploy the app on Streamlit Community Cloud

12
Automated GitHub Actions Workflow	- Create a pipeline.yml file in the GitHub repo
- Define a YAML file and list all steps to execute each step of Machine Learning

- Push all files to GitHub

- Automate the end-to-end workflow

- Update the workflow to automatically push code updates to the main branch

15
Output Evaluation	- GitHub (link to repository, screenshot of folder structure and executed workflow)
- Streamlit Community Cloud (link to the Streamlit app, screenshot of Streamlit app)

8
Actionable Insights and Recommendations	- Key takeaways for the business	4
Business Report Quality	- Adhere to the business report checklist	6
Total		60
